<a href="https://colab.research.google.com/github/ruforavishnu/Project_Machine_Learning/blob/master/Project21_Reinforcement_Learning_Autonomous_Car_Parking_using_Deep_Q_Networks_(DQN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install gym torch torchvision matplotlib numpy tqdm

In [2]:
pip install pygame

In [3]:
import gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, patches
import math, random
from tqdm import tqdm

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
import tensorflow as tf
from tensorflow.keras import layers

In [5]:
class ParkingEnv(gym.Env):
  def __init__(self):
    super().__init__()

    self.dt = 0.2
    self.max_speed = 3.0
    self.max_steer = math.pi/6

    self.action_space = gym.spaces.Discrete(5)


    high = np.array([100,100, math.pi, 5, 100,100], dtype=np.float32)

    self.observation_space = gym.spaces.Box(-high, high)


    self.reset()




  def reset(self):
    self.x = np.random.uniform(0,2)
    self.y = np.random.uniform(0,2)
    self.angle = np.random.uniform( -math.pi,  math.pi)
    self.v = 0.0


    self.goal_x =  np.random.uniform(6,8)
    self.goal_y = np.random.uniform(6,8)

    self.t = 0

    return self._get_state()




  def _get_state(self):
    return np.array([self.x, self.y, self.angle, self.v, self.goal_x, self.goal_y],  dtype=np.float32)





  def step(self, action):
    accel = 0
    steer = 0


    if action == 0: accel = 1.0
    elif action == 1: accel = -2.0
    elif action == 2: steer = self.max_steer/2
    elif action == 3: steer = -self.max_steer/2
    elif action == 4: accel = 0.4



    self.v += accel * self.dt
    self.v = np.clip(self.v, -self.max_speed, self.max_speed)

    self.angle += steer * self.dt
    self.x += self.v * math.cos(self.angle) * self.dt
    self.y += self.v * math.sin(self.angle) * self.dt



    dist = math.hypot(self.x - self.goal_x,  self.y - self.goal_y)
    reward = -dist * 0.1
    done = False




    if dist < 0.5 and abs(self.v) < 0.4:
      reward += 40
      done = True


    if not (0 <= self.x <= 10     and     0 <= self.y <= 10):
      reward -= 20
      done = True

    self.t += 1
    if self.t >= 200:
      done = True



    return self._get_state(), reward, done, {}





### Build the TensorFlow DQN Network

In [6]:
def create_q_model(state_dim,  action_dim):
  model = tf.keras.Sequential([
      layers.Input(shape=(state_dim,)),
      layers.Dense(128, activation='relu'),
      layers.Dense(128, activation='relu'),
      layers.Dense(action_dim)


  ])

  return model


### Setup Agent + Replay Buffer

In [ ]:
class ReplayBuffer:
  def __init__(self, size=200000):

